# ???????? ?????? ????????? ???????? ? ???????? ?? ????????? ????

???? ??????? ???????? ??? ??? ???????? ?????? ????????? ???????? ? ???????? ?? ????????? ????. ?????? ???????? ?? ??????????? MarianMT ?? ?????????? `transformers` ? ????????? ?? ???????????????? ?????? ?????? `russian_aleut_dataset.csv`, ?????????? ???? ?????????.

## ??????????
- Python 3.11 (?????????????).
- Google Colab ? ???????? ? GPU (??? ????????? ????????).
- ??????? `russian_aleut_dataset.csv` (?????? ???? ???????? ? Colab).

## ???????????
????????? ??????????? ??????????:

In [ ]:
!pip install transformers torch pandas datasets

## ?????? ????????? ? ????????? ??????????

??????????? ??????????? ?????????? ? ?????????? ?????????? (GPU ??? CPU) ??? ????????.

In [ ]:
import pandas as pd
import torch
from datasets import Dataset
from google.colab import files
from transformers import (
    MarianMTModel,
    MarianTokenizer,
    Trainer,
    TrainingArguments,
)

# ?????????? ?????????? ??? ???????? (GPU ??? CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"???????????? ??????????: {device}")

## ???????? ? ?????????? ??????

????????? ??????? ?? ????? `russian_aleut_dataset.csv` ? ??????????? ??? ? ?????? `Dataset` ??? ?????? ? ??????????? `transformers`.

In [ ]:
def load_dataset(file_path: str) -> Dataset:
    """????????? ??????? ?? CSV-????? ? ??????????? ??? ? ?????? Dataset.

    Args:
        file_path: ???? ? CSV-????? ? ??????? (???????-?????????).

    Returns:
        Dataset: ????? ?????? ? ??????? Hugging Face Dataset.
    """
    # ?????? CSV-???? ? ?????? ?????????
    df = pd.read_csv(file_path, sep=";", encoding="utf-8")
    translation_pairs = df[["Russian", "Aleut"]].dropna().to_dict("records")

    # ??????????? ? ?????? Dataset
    dataset = Dataset.from_list(
        [{"source": pair["Russian"], "target": pair["Aleut"]} for pair in translation_pairs]
    )
    return dataset

# ????????? ???????
dataset = load_dataset("russian_aleut_dataset.csv")

## ????????????? ?????? ? ????????????

?????????????? ????????????? ?????? MarianMT ? ???????????, ????????? ?????????? `?`, ??????????? ??? ?????????? ?????.

In [ ]:
def initialize_model_and_tokenizer(model_name: str) -> tuple[MarianMTModel, MarianTokenizer]:
    """?????????????? ?????? ? ??????????? MarianMT."""
    
    # ????????? ??????????? ? ??????
    tokenizer = MarianTokenizer.from_pretrained(model_name)
    model = MarianMTModel.from_pretrained(model_name).to(device)

    # ????????? ?????????? "?" ? ??????????? ? ????????? ?????? ???????????
    tokenizer.add_tokens(["?"])
    model.resize_token_embeddings(len(tokenizer))

    # ????????? ??????????? ? ????????????? ?? ???????? ??????
    sample_text = "qana-? angali-? ula-? a-ku-?"
    encoded = tokenizer(sample_text, return_tensors="pt")
    decoded = tokenizer.decode(encoded["input_ids"][0], skip_special_tokens=True)
    print("?????????????? ? ????????????:", decoded)

    return model, tokenizer

# ?????????????? ?????? ? ???????????
model_name = "Helsinki-NLP/opus-mt-ru-en"
model, tokenizer = initialize_model_and_tokenizer(model_name)

## ????????????? ??????

???????????? ??????? (???????) ? ??????? (?????????) ??????, ????????????? ?? ??? ????????.

In [ ]:
def preprocess_function(examples: dict) -> dict:
    """?????????????? ?????? ??? ????????: ???????????? ??????? ? ??????? ??????."""
    
    inputs = examples["source"]
    targets = examples["target"]

    # ???????????? ??????? ?????? (???????)
    model_inputs = tokenizer(
        inputs,
        max_length=128,
        truncation=True,
        padding="max_length",
    )

    # ???????????? ??????? ?????? (?????????) ? ?????? ???????? ????????????
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            targets,
            max_length=128,
            truncation=True,
            padding="max_length",
        )

    # ????????? ????? (labels) ? ???????? ???????
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# ????????? ????????????? ? ????????
tokenized_dataset = dataset.map(preprocess_function, batched=True)

# ????????? ?? ????????? ? ???????? ??????? (90% ? ????????, 10% ? ????)
train_test = tokenized_dataset.train_test_split(test_size=0.1)
train_dataset = train_test["train"]
eval_dataset = train_test["test"]

## ???????? ??????

??????????? ????????? ???????? ? ????????? ??????? ?????????? ??????.

In [ ]:
def train_model(
    model: MarianMTModel,
    tokenizer: MarianTokenizer,
    train_dataset: Dataset,
    eval_dataset: Dataset,
) -> None:
    """??????? ?????? ?? ?????????????? ??????."""
    
    # ??????????? ????????? ????????
    training_args = TrainingArguments(
        output_dir="./mbart_aleut",
        evaluation_strategy="epoch",
        learning_rate=5e-5,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=2,
        num_train_epochs=5,
        weight_decay=0.01,
        save_strategy="epoch",
        load_best_model_at_end=True,
        logging_dir="./logs",
        logging_steps=10,
        report_to="none",
        fp16=True,
    )

    # ?????????????? Trainer ??? ????????
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
    )

    # ????????? ????????
    trainer.train()

# ??????? ??????
train_model(model, tokenizer, train_dataset, eval_dataset)

## ?????????? ??????

????????? ????????? ?????? ? ??????????? ? ????? `Marian_aleut_model`, ?????????? ?? ? ????????? ?????.

In [ ]:
def save_model_and_tokenizer(
    model: MarianMTModel,
    tokenizer: MarianTokenizer,
    save_dir: str,
    zip_name: str,
) -> None:
    """????????? ?????? ? ???????????, ? ????? ?????????? ?? ? zip-????."""
    
    # ????????? ?????? ? ???????????
    model.save_pretrained(save_dir)
    tokenizer.save_pretrained(save_dir)

    # ?????????? ????? ? ???????
    !zip -r {zip_name} {save_dir}

    # ????????? ?????
    files.download(zip_name)

# ????????? ??????
save_model_and_tokenizer(model, tokenizer, "Marian_aleut_model", "Marian_aleut_model.zip")

## ???????????? ??????

????????? ?????? ?????? ?? ?????????? ???????? ??????.

In [ ]:
def translate(text: str, model: MarianMTModel, tokenizer: MarianTokenizer) -> str:
    """????????? ????? ? ???????? ?? ????????? ? ?????????????? ????????? ??????."""
    
    # ???????????? ??????? ?????
    inputs = tokenizer(text, return_tensors="pt", padding=True).to(device)

    # ?????????? ???????
    translated_tokens = model.generate(
        **inputs,
        max_length=128,
        num_beams=5,
        early_stopping=True,
    )

    # ?????????? ?????????
    result = tokenizer.decode(translated_tokens[0], skip_special_tokens=True)

    # ?????????? ??????? ????? ???????????? "?"
    result = result.replace(" ?", "?")
    return result

# ????????? ??????
print("???????????? ??????:")
test_phrases = ["??? ??????? ????", "??? ????? ?????"]
for phrase in test_phrases:
    translation = translate(phrase, model, tokenizer)
    print(f"???????: {phrase} -> ?????????: {translation}")